# Student Performance and Attendance Analysis
This notebook runs the documented cleaning pipeline, explores cleaned synthetic records and calculates evidence-based academic KPIs. Synthetic results are not institutional outcomes.

## Problem statement and project scope
Marks and attendance records are often held separately. This notebook demonstrates how to validate and combine them to report student performance, pass percentage, subject outcomes, attendance risk and rankings. The committed records are synthetic and fictional, not institutional results.

## Dataset description
The reproducible sample contains 200 fictional students, five subjects, 1,000 student-subject marks records, matching attendance records and one academic term. Raw files intentionally include duplicate, missing, inconsistent and invalid examples. The cleaning pipeline quarantines rejected records and keeps ABSENT separate from MISSING.


## 1. Import libraries and configure paths
Import only the libraries used for data processing and plotting.

In [ ]:
from pathlib import Path
import runpy
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
CLEAN = ROOT / 'data' / 'clean'
print('Project root:', ROOT)

## 2. Run the documented cleaning pipeline
This ensures notebook analysis uses the same cleaning rules and quarantine outputs as the command-line pipeline.

In [ ]:
runpy.run_path(str(ROOT / 'src' / '02_clean_data.py'), run_name='__main__')

## 3. Load cleaned datasets
Load the cleaned student, subject, marks and attendance files.

In [ ]:
students = pd.read_csv(CLEAN / 'students.csv')
subjects = pd.read_csv(CLEAN / 'subjects.csv')
marks = pd.read_csv(CLEAN / 'marks.csv')
attendance = pd.read_csv(CLEAN / 'attendance.csv')
scored = marks.loc[marks['Record_Status'].eq('PRESENT')].copy()
scored['Total_Marks'] = pd.to_numeric(scored['Total_Marks'], errors='coerce')
display(students.head(), subjects.head(), marks.head(), attendance.head())

## 4. Dataset profile and data quality
Review dimensions and the cleaning log so exclusions and corrections remain visible.

In [ ]:
print('Students:', students['Student_ID'].nunique())
print('Subjects:', subjects['Subject_ID'].nunique())
print('Marks rows:', len(marks), '| scored:', len(scored))
print('Attendance rows:', len(attendance))
quality_log = pd.read_csv(ROOT / 'reports' / 'data_quality_log.csv')
display(quality_log)

## 5. Core KPIs
Pass percentage uses PRESENT scored student-subject rows as its denominator. ABSENT and MISSING are reported separately and are not converted to zero.

In [ ]:
pass_count = int(scored['Result'].eq('PASS').sum())
fail_count = int(scored['Result'].eq('FAIL').sum())
pass_rate = pass_count / len(scored) * 100 if len(scored) else 0
average_marks = np.nanmean(scored['Total_Marks'].to_numpy())
average_attendance = attendance['Attendance_Percentage'].mean()
import json
rules = json.loads((ROOT / 'config' / 'business_rules.json').read_text(encoding='utf-8'))
low_threshold = float(rules['attendance_threshold_pct'])
student_attendance = attendance.groupby('Student_ID')['Attendance_Percentage'].mean()
low_attendance_count = int((student_attendance < low_threshold).sum())
kpis = pd.Series({'Average marks': average_marks, 'Average attendance (%)': average_attendance,
                  'Pass percentage': pass_rate, 'Pass records': pass_count,
                  'Fail records': fail_count, 'Low-attendance students': low_attendance_count})
display(kpis.round(2))

## 6. Marks distribution
A histogram shows the distribution of valid scored records.

In [ ]:
sns.histplot(data=scored, x='Total_Marks', bins=15)
plt.title('Distribution of Student Marks')
plt.xlabel('Total marks out of 100')
plt.ylabel('Student-subject records')
plt.show()

## 7. Subject-wise performance
Compare average marks by subject using scored records only.

In [ ]:
subject_summary = scored.merge(subjects[['Subject_ID','Subject_Name']], on='Subject_ID')
subject_summary = subject_summary.groupby('Subject_Name')['Total_Marks'].mean().sort_values()
display(subject_summary.to_frame('Average_Marks'))
subject_summary.plot(kind='barh', figsize=(8, 4), title='Subject-wise Average Marks', xlabel='Average marks')
plt.show()

## 8. Attendance and marks relationship
Join at the common student-subject grain. This is not evidence of causation.

In [ ]:
relationship = scored.merge(attendance[['Student_ID','Subject_ID','Term_ID','Attendance_Percentage']],
                            on=['Student_ID','Subject_ID','Term_ID'], validate='one_to_one')
display(relationship[['Attendance_Percentage','Total_Marks']].corr().round(3))
plt.figure(figsize=(7, 4))
sns.scatterplot(data=relationship, x='Attendance_Percentage', y='Total_Marks', alpha=.6)
plt.title('Attendance vs Academic Marks')
plt.xlabel('Attendance (%)'); plt.ylabel('Total marks'); plt.show()

## 9. Top students and low attendance
Tied aggregate marks are not broken conceptually; dense ranking should be used in SQL/Power BI.

In [ ]:
student_totals = scored.groupby('Student_ID', as_index=False)['Total_Marks'].sum()
student_totals = student_totals.merge(students[['Student_ID','Student_Name']], on='Student_ID')
student_totals['Rank'] = student_totals['Total_Marks'].rank(method='dense', ascending=False).astype(int)
display(student_totals.sort_values(['Rank','Student_ID']).head(10))
low_attendance = student_attendance[student_attendance < low_threshold].sort_values()
display(low_attendance.rename('Average_Attendance_Percentage').to_frame())

## 10. Findings and recommendations
The following values are computed from the current synthetic dataset at notebook execution time. Do not copy them into an institutional report without replacing the dataset with authorised records.

In [ ]:
print(f'Average marks: {average_marks:.2f}')
print(f'Average attendance: {average_attendance:.2f}%')
print(f'Pass percentage among scored records: {pass_rate:.2f}%')
print(f'Low-attendance students (<{low_threshold:.1f}% provisional): {low_attendance_count}')
print('Recommendation: review low-attendance students with faculty and inspect subject-level failure patterns.')
print('Limitation: synthetic observational data cannot establish that attendance causes marks.')